# 02 — Analysis

Owner: Dat (Q1, Q3) + Khoi (Q4). Data loaded from `data/processed/` (see ADR-003).

Authority order: `ADR > SPECIFICATION.md > notebook`.

In [1]:
# Setup — imports, paths, constants
import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
from scipy.stats import ConstantInputWarning
warnings.filterwarnings("ignore", category=ConstantInputWarning)

# Repo root on sys.path so we can reuse prepare_data.py constants
ROOT = Path("..").resolve() if not Path("data").exists() else Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from prepare_data import EDGAR_COUNTRY_MAP, SECTOR_GROUP, TARGET_COUNTRIES

DATA = ROOT / "data"
PROCESSED = DATA / "processed"

SEED = 20261123
RNG = np.random.default_rng(SEED)

COUNTRY_ISO3 = {
    "Vietnam": "VNM", "Thailand": "THA", "Singapore": "SGP",
    "China": "CHN", "Japan": "JPN", "Germany": "DEU",
    "United States": "USA", "New Zealand": "NZL",
}

SPEARMAN_CRIT = 0.7857  # displayed as 0.786 (SPECIFICATION S5)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

print("Setup OK — data dir:", DATA.resolve())


Setup OK — data dir: /home/datle/programming-for-ds/csc17104-ev-adoption-grid/data


## Q1 — Transport share vs adoption pace (Dat)

**Question (ADR-001, SPECIFICATION S1):** Does the transport weight in national CO2 predict the pace of electrification, after controlling for income?

**Hypothesis H1 (directional):** higher transport share goes with faster adoption. If the sign is opposite or indistinguishable from 0, report as-is (no causal language).

**Sample:** 7 countries (Singapore excluded — no IEA rows). Transport share from EDGAR 2025 (structure described for all 8).

**Decision rule (SPECIFICATION S5):** |rho| >= 0.786 (exact 0.7857, alpha=0.05 two-sided, 5,040 permutations) **and** bootstrap 95% CI excludes 0. With n=7, a null result is **not** evidence of no relationship.

**Depth criteria:** D1 (three-source merge), D2 (period-over-period change), D3 (transport CO2 per capita, adoption pace), D4 (income control), D5 (bootstrap CI), D6 (year & series robustness).


In [2]:
# --- Load processed CSVs (ADR-003) ---
emissions = pd.read_csv(PROCESSED / "emissions_long.csv")
ev = pd.read_csv(PROCESSED / "ev_long.csv")

print("emissions_long:", emissions.shape, "rows,", emissions["country"].nunique(), "countries")
print("ev_long:", ev.shape, "rows,", ev["country"].nunique(), "countries")
assert emissions.shape[0] == 61, f"expected 61 emissions rows, got {len(emissions)}"
assert ev.shape[0] == 496, f"expected 496 ev rows, got {len(ev)}"

print("EDGAR years:", sorted(emissions["year"].unique()))
print("EV years:", int(ev["year"].min()), "to", int(ev["year"].max()))
print("EV parameters:", sorted(ev["parameter"].unique()))
assert set(ev["parameter"].unique()) == {"EV sales share", "EV stock share", "EV charging points"}


emissions_long: (61, 6) rows, 8 countries
ev_long: (496, 7) rows, 7 countries
EDGAR years: [2025]
EV years: 2010 to 2024
EV parameters: ['EV charging points', 'EV sales share', 'EV stock share']


In [3]:
# --- EDGAR 2025: transport & sector shares per country ---
co2 = emissions[(emissions["substance"] == "CO2") & (emissions["year"] == 2025)].copy()

totals = co2.groupby("country")["emissions_mt"].sum().rename("total_mt")
transport = co2[co2["sector"] == "Transport"].set_index("country")["emissions_mt"].rename("transport_mt")

sector_pivot = co2.pivot_table(index="country", columns="sector_group",
                               values="emissions_mt", aggfunc="sum", fill_value=0)
sector_shares = sector_pivot.div(sector_pivot.sum(axis=1), axis=0) * 100

q1 = pd.DataFrame({"transport_mt": transport, "total_mt": totals})
for col in ["Transport", "Industry", "Power", "Other"]:
    if col in sector_shares.columns:
        q1[f"{col.lower()}_share"] = sector_shares[col]

q1 = q1.sort_index()
print("Transport share of national CO2, 2025 (8 countries):")
display(q1.round(2))


Transport share of national CO2, 2025 (8 countries):


,transport_mt,total_mt,transport_share,industry_share,power_share,other_share
country,,,,,,
China,1088.25,13057.11,8.33,30.74,50.28,10.64
Germany,136.87,567.76,24.11,20.42,31.43,24.04
Japan,179.84,977.28,18.40,21.57,44.89,15.14
New Zealand,15.38,29.16,52.73,21.22,11.91,14.13
Singapore,5.73,57.14,10.02,41.68,39.50,8.81
Thailand,81.00,266.94,30.34,27.27,30.59,11.80
United States,1724.64,4739.09,36.39,12.96,33.03,17.63
Vietnam,50.81,386.97,13.13,35.57,46.04,5.26


In [4]:
# --- IEA: adoption pace (cars, 2018 -> 2024) ---
ev_sales = ev[(ev["parameter"] == "EV sales share") & (ev["mode"] == "Cars")
              & (ev["powertrain"] == "EV")].copy()

level_2018 = ev_sales[ev_sales["year"] == 2018].set_index("country")["value"].rename("ev_share_2018")
level_2024 = ev_sales[ev_sales["year"] == 2024].set_index("country")["value"].rename("ev_share_2024")

pace = pd.DataFrame({"ev_share_2018": level_2018, "ev_share_2024": level_2024})
pace["pace_pp"] = pace["ev_share_2024"] - pace["ev_share_2018"]

print("EV sales share (cars), 2018 vs 2024, and pace (pp):")
display(pace.round(4))

edgar_countries = set(q1.index)
iea_countries = set(pace.index)
print(f"\nEDGAR countries: {sorted(edgar_countries)}")
print(f"IEA countries (cars, 2018 & 2024): {sorted(iea_countries)}")
print(f"Excluded (no IEA data): {sorted(edgar_countries - iea_countries)}")
print(f"Unmatched IEA -> EDGAR: {sorted(iea_countries - edgar_countries)}")


EV sales share (cars), 2018 vs 2024, and pace (pp):


,ev_share_2018,ev_share_2024,pace_pp
country,,,
China,4.7000,48.0,43.3000
Germany,2.0000,19.0,17.0000
Japan,1.1000,2.8,1.7000
New Zealand,4.9000,11.0,6.1000
Thailand,0.8700,13.0,12.1300
United States,2.3000,10.0,7.7000
Vietnam,0.0016,17.0,16.9984



EDGAR countries: ['China', 'Germany', 'Japan', 'New Zealand', 'Singapore', 'Thailand', 'United States', 'Vietnam']
IEA countries (cars, 2018 & 2024): ['China', 'Germany', 'Japan', 'New Zealand', 'Thailand', 'United States', 'Vietnam']
Excluded (no IEA data): ['Singapore']
Unmatched IEA -> EDGAR: []


In [5]:
# --- World Bank Open Data: GDP per capita, population, income group ---
# Source: https://data.worldbank.org/ ; CC BY 4.0; accessed at runtime.
# Indicators: NY.GDP.PCAP.CD (GDP per capita), SP.POP.TOTL (population),
# incomeLevel (country endpoint). Cache to data/processed/ for reproducibility.
WB_CACHE = PROCESSED / "world_bank_q1.json"
ACCESS_DATE = "2026-11-24"

def fetch_wb():
    iso_list = ";".join(COUNTRY_ISO3.values())
    gdp_url = (f"https://api.worldbank.org/v2/country/{iso_list}/indicator/NY.GDP.PCAP.CD"
               f"?format=json&date=2018:2024&per_page=1000")
    pop_url = (f"https://api.worldbank.org/v2/country/{iso_list}/indicator/SP.POP.TOTL"
               f"?format=json&date=2024&per_page=1000")
    country_url = f"https://api.worldbank.org/v2/country/{iso_list}?format=json&per_page=1000"

    gdp_resp = requests.get(gdp_url, timeout=30).json()
    pop_resp = requests.get(pop_url, timeout=30).json()
    country_resp = requests.get(country_url, timeout=30).json()

    records = {
        "gdp": gdp_resp[1],
        "pop": pop_resp[1],
        "country": country_resp[1],
        "access_date": ACCESS_DATE,
        "last_updated": gdp_resp[0].get("lastupdated"),
    }
    with open(WB_CACHE, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"Cached World Bank data to {WB_CACHE}")
    return records

if WB_CACHE.exists():
    with open(WB_CACHE, encoding="utf-8") as f:
        wb = json.load(f)
    print(f"Loaded cached World Bank data from {WB_CACHE}")
else:
    wb = fetch_wb()

# Parse GDP per capita
gdp_map = {}
for r in wb["gdp"]:
    iso3 = r["countryiso3code"]
    year = r["date"]
    val = r["value"]
    if val is not None:
        gdp_map.setdefault(iso3, {})[year] = val

# Parse population (2024)
pop_map = {}
for r in wb["pop"]:
    iso3 = r["countryiso3code"]
    if r["value"] is not None:
        pop_map[iso3] = r["value"]

# Parse income group
income_map = {}
for c in wb["country"]:
    income_map[c["id"]] = c["incomeLevel"]["value"]

wb_df = pd.DataFrame({"iso3": pd.Series(COUNTRY_ISO3)})
wb_df["gdp_2018"] = wb_df["iso3"].map(lambda x: gdp_map.get(x, {}).get("2018"))
wb_df["gdp_2024"] = wb_df["iso3"].map(lambda x: gdp_map.get(x, {}).get("2024"))
wb_df["pop_2024"] = wb_df["iso3"].map(pop_map)
wb_df["income_group"] = wb_df["iso3"].map(income_map)
wb_df.index.name = "country"

print(f"\nWorld Bank: access {wb['access_date']}, API lastupdated {wb.get('last_updated')}")
print("World Bank controls:")
display(wb_df.round(0))


Loaded cached World Bank data from /home/datle/programming-for-ds/csc17104-ev-adoption-grid/data/processed/world_bank_q1.json

World Bank: access 2026-11-24, API lastupdated 2026-11-11
World Bank controls:


,iso3,gdp_2018,gdp_2024,pop_2024,income_group
country,,,,,
Vietnam,VNM,3222.0,4717.0,100987686,Upper middle income
Thailand,THA,7100.0,7387.0,71668011,Upper middle income
Singapore,SGP,67033.0,94897.0,6036860,High income
China,CHN,10086.0,13293.0,1408975000,Upper middle income
Japan,JPN,40645.0,33797.0,123975371,High income
Germany,DEU,48916.0,56104.0,83516593,High income
United States,USA,62876.0,86170.0,340003797,High income
New Zealand,NZL,43204.0,49432.0,5290000,High income


In [6]:
# --- Merge: EDGAR (8) x IEA (7) x World Bank (8) -> 7 ---
q1_full = q1.join(pace, how="inner")
q1_full = q1_full.join(wb_df[["gdp_2018", "gdp_2024", "pop_2024", "income_group"]], how="inner")

# Derived variables
q1_full["transport_t_per_capita"] = q1_full["transport_mt"] * 1e6 / q1_full["pop_2024"]
q1_full["log_gdp_2018"] = np.log(q1_full["gdp_2018"])
q1_full["log_gdp_2024"] = np.log(q1_full["gdp_2024"])

# Merge audit
edgar_set = set(q1.index)
iea_set = set(pace.index)
wb_set = set(wb_df.index)
merged_set = set(q1_full.index)

print("=== Merge audit ===")
print(f"EDGAR countries (transport share): {len(edgar_set)} -> {sorted(edgar_set)}")
print(f"IEA countries (cars, 2018 & 2024): {len(iea_set)} -> {sorted(iea_set)}")
print(f"World Bank countries: {len(wb_set)} -> {sorted(wb_set)}")
print(f"Merged (analysis sample): {len(merged_set)} -> {sorted(merged_set)}")
print(f"EDGAR - IEA (excluded by design): {sorted(edgar_set - iea_set)}")
print(f"IEA - EDGAR (unexpected): {sorted(iea_set - edgar_set)}")
print(f"WB - merged (unexpected): {sorted(wb_set - merged_set)}")

assert len(merged_set) == 7, f"expected 7, got {len(merged_set)}"
assert edgar_set - iea_set == {"Singapore"}, "unexpected EDGAR-IEA mismatch"
assert iea_set - edgar_set == set(), "unexpected IEA-EDGAR mismatch"
assert merged_set == iea_set, "merge dropped an IEA country"

print("\nQ1 analysis table:")
display_cols = ["transport_share", "industry_share", "power_share", "other_share",
                "transport_mt", "transport_t_per_capita",
                "ev_share_2018", "ev_share_2024", "pace_pp",
                "gdp_2024", "income_group"]
display(q1_full[display_cols].round(3))


=== Merge audit ===
EDGAR countries (transport share): 8 -> ['China', 'Germany', 'Japan', 'New Zealand', 'Singapore', 'Thailand', 'United States', 'Vietnam']
IEA countries (cars, 2018 & 2024): 7 -> ['China', 'Germany', 'Japan', 'New Zealand', 'Thailand', 'United States', 'Vietnam']
World Bank countries: 8 -> ['China', 'Germany', 'Japan', 'New Zealand', 'Singapore', 'Thailand', 'United States', 'Vietnam']
Merged (analysis sample): 7 -> ['China', 'Germany', 'Japan', 'New Zealand', 'Thailand', 'United States', 'Vietnam']
EDGAR - IEA (excluded by design): ['Singapore']
IEA - EDGAR (unexpected): []
WB - merged (unexpected): ['Singapore']

Q1 analysis table:


,transport_share,industry_share,power_share,other_share,transport_mt,transport_t_per_capita,ev_share_2018,ev_share_2024,pace_pp,gdp_2024,income_group
country,,,,,,,,,,,
China,8.335,30.741,50.279,10.645,1088.255,0.772,4.700,48.0,43.300,13293.116,Upper middle income
Germany,24.108,20.424,31.426,24.043,136.874,1.639,2.000,19.0,17.000,56103.732,High income
Japan,18.402,21.569,44.892,15.137,179.838,1.451,1.100,2.8,1.700,33797.101,High income
New Zealand,52.733,21.224,11.912,14.132,15.379,2.907,4.900,11.0,6.100,49432.363,High income
Thailand,30.344,27.267,30.589,11.800,81.001,1.130,0.870,13.0,12.130,7386.636,Upper middle income
United States,36.392,12.957,33.025,17.626,1724.636,5.072,2.300,10.0,7.700,86169.664,High income
Vietnam,13.131,35.565,46.041,5.263,50.812,0.503,0.002,17.0,16.998,4716.660,Upper middle income


In [7]:
# --- Primary analysis: Spearman rank correlation ---
x = q1_full["transport_share"].values
y = q1_full["pace_pp"].values
n = len(x)

rho, p_asym = stats.spearmanr(x, y)
print(f"Spearman rho = {rho:.4f}, asymptotic p = {p_asym:.4f}, n = {n}")

# Exact permutation p-value (5,040 permutations for n=7)
from itertools import permutations
observed_abs = abs(rho)
count = 0
total = 0
for perm in permutations(range(n)):
    r, _ = stats.spearmanr(x, y[list(perm)])
    total += 1
    if abs(r) >= observed_abs - 1e-12:
        count += 1
p_exact = count / total
print(f"Exact permutation p (two-sided, {total} reps) = {p_exact:.4f}")

# Bootstrap 95% CI (10,000 resamples)
rhos = []
for _ in range(10000):
    idx = RNG.integers(0, n, size=n)
    r, _ = stats.spearmanr(x[idx], y[idx])
    rhos.append(r)
rhos = np.array(rhos)
ci_lo, ci_hi = np.percentile(rhos, [2.5, 97.5])
print(f"Bootstrap 95% CI (10,000 resamples): [{ci_lo:.4f}, {ci_hi:.4f}]")
print(f"Bootstrap mean rho: {rhos.mean():.4f}, std: {rhos.std():.4f}")

# Decision rule
meets_threshold = abs(rho) >= SPEARMAN_CRIT
ci_excludes_zero = (ci_lo > 0) or (ci_hi < 0)
print(f"\n=== Decision rule (SPECIFICATION S5) ===")
print(f"|rho| = {abs(rho):.4f} >= {SPEARMAN_CRIT:.4f} (0.786)? {meets_threshold}")
print(f"Bootstrap CI excludes 0? {ci_excludes_zero}")
print(f"Decision: {'MEETS threshold' if (meets_threshold and ci_excludes_zero) else 'Does NOT meet threshold'}")


Spearman rho = -0.5714, asymptotic p = 0.1802, n = 7
Exact permutation p (two-sided, 5040 reps) = 0.2000
Bootstrap 95% CI (10,000 resamples): [-1.0000, 0.3725]
Bootstrap mean rho: -0.5563, std: 0.3965

=== Decision rule (SPECIFICATION S5) ===
|rho| = 0.5714 >= 0.7857 (0.786)? False
Bootstrap CI excludes 0? False
Decision: Does NOT meet threshold


In [8]:
# --- Income control: OLS with log GDP + income group subgroups ---
# Main control: log GDP per capita 2018 (pre-period baseline)
model_2018 = smf.ols("pace_pp ~ transport_share + log_gdp_2018", data=q1_full).fit()
print("OLS: pace_pp ~ transport_share + log_gdp_2018")
print(model_2018.summary().as_text())
print("\nCoefficients:")
print(model_2018.params.round(4))
print("\n95% CI:")
print(model_2018.conf_int().round(4))

# Robustness: log GDP 2024
model_2024 = smf.ols("pace_pp ~ transport_share + log_gdp_2024", data=q1_full).fit()
print("\n--- Robustness: pace_pp ~ transport_share + log_gdp_2024 ---")
print("transport_share coef:", round(model_2024.params["transport_share"], 4),
      "CI:", model_2024.conf_int().loc["transport_share"].round(4).tolist(),
      "p:", round(model_2024.pvalues["transport_share"], 4))

# Income group subgroups (descriptive; n too small for within-group tests)
print("\n=== Income group subgroups ===")
for grp, sub in q1_full.groupby("income_group"):
    print(f"\n{grp} (n={len(sub)}):")
    print(f"  transport_share: mean={sub['transport_share'].mean():.1f}, "
          f"median={sub['transport_share'].median():.1f}")
    print(f"  pace_pp: mean={sub['pace_pp'].mean():.2f}, median={sub['pace_pp'].median():.2f}")
    if len(sub) >= 4:
        r, p = stats.spearmanr(sub["transport_share"], sub["pace_pp"])
        print(f"  Spearman within group: rho={r:.3f}, p={p:.3f}")


OLS: pace_pp ~ transport_share + log_gdp_2018
                            OLS Regression Results                            
Dep. Variable:                pace_pp   R-squared:                       0.388
Model:                            OLS   Adj. R-squared:                  0.083
Method:                 Least Squares   F-statistic:                     1.270
Date:                Tue, 24 Nov 2026   Prob (F-statistic):              0.374
Time:                        22:30:57   Log-Likelihood:                -25.995
No. Observations:                   7   AIC:                             57.99
Df Residuals:                       4   BIC:                             57.83
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------

/home/datle/programming-for-ds/.venv/lib/python3.11/site-packages/statsmodels/stats/stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 7 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


In [9]:
# --- Plots ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1: scatter transport share vs pace
ax = axes[0]
color_map = {"High income": "#1f77b4", "Upper middle income": "#ff7f0e"}
colors = q1_full["income_group"].map(color_map)
ax.scatter(q1_full["transport_share"], q1_full["pace_pp"], c=colors, s=90,
           edgecolor="k", zorder=3, linewidth=0.5)
for country, row in q1_full.iterrows():
    ax.annotate(country, (row["transport_share"], row["pace_pp"]),
                textcoords="offset points", xytext=(6, 4), fontsize=9)
# OLS fit line (GDP 2018 control)
x_line = np.linspace(q1_full["transport_share"].min() - 2,
                     q1_full["transport_share"].max() + 2, 100)
X_pred = pd.DataFrame({"transport_share": x_line,
                       "log_gdp_2018": q1_full["log_gdp_2018"].mean()})
pred = model_2018.get_prediction(X_pred).summary_frame(alpha=0.05)
ax.plot(x_line, pred["mean"], "r-", lw=1.5, label="OLS fit (GDP 2018 control)")
ax.fill_between(x_line, pred["mean_ci_lower"], pred["mean_ci_upper"],
                color="r", alpha=0.15, label="95% CI")
ax.set_xlabel("Transport share of national CO2, 2025 (%)")
ax.set_ylabel("EV sales share change, cars, 2018->2024 (pp)")
ax.set_title("Q1: Transport share vs adoption pace")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Plot 2: adoption pace by country (sorted), highlight China
ax = axes[1]
pace_sorted = q1_full.sort_values("pace_pp")
bar_colors = ["#d62728" if c == "China" else "#1f77b4" for c in pace_sorted.index]
ax.barh(pace_sorted.index, pace_sorted["pace_pp"], color=bar_colors, height=0.7)
ax.set_xlabel("EV sales share change, cars, 2018->2024 (pp)")
ax.set_title("Adoption pace by country")
ax.grid(True, axis="x", alpha=0.3)
for i, (country, row) in enumerate(pace_sorted.iterrows()):
    ax.text(row["pace_pp"] + 0.3, i, f"{row['pace_pp']:.1f}", va="center", fontsize=8)

plt.tight_layout()
plt.show()


<Figure size 1430x550 with 2 Axes>

In [10]:
# --- Robustness checks (SPECIFICATION S6) ---

def edgar_transport_share(year):
    """Compute transport share per country from raw EDGAR for a given year."""
    raw = pd.read_excel(DATA / "EDGAR_2026_GHG_booklet_2026.xlsx",
                        sheet_name="GHG_by_sector_and_country", engine="openpyxl")
    ycol = str(year) if str(year) in raw.columns else year
    df = raw.loc[raw["Substance"] == "CO2", ["Country", "Sector", ycol]].copy()
    df = df.rename(columns={ycol: "emissions_mt", "Country": "src_country", "Sector": "sector"})
    df = df[df["src_country"].isin(EDGAR_COUNTRY_MAP)].copy()
    df["country"] = df["src_country"].map(EDGAR_COUNTRY_MAP)
    totals = df.groupby("country")["emissions_mt"].sum()
    transport = df[df["sector"] == "Transport"].set_index("country")["emissions_mt"]
    return (transport / totals * 100)

def spearman_with_ci(x, y, n_boot=10000, seed=SEED):
    """Return (rho, ci_lo, ci_hi, n, n_valid) for a given (x, y) pair.
    Bootstrap resamples that yield NaN Spearman (constant input) are skipped."""
    rng = np.random.default_rng(seed)
    rho, _ = stats.spearmanr(x, y)
    rhos = []
    n = len(x)
    n_skip = 0
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        r, _ = stats.spearmanr(x[idx], y[idx])
        if np.isnan(r):
            n_skip += 1
            continue
        rhos.append(r)
    rhos = np.array(rhos)
    ci = np.percentile(rhos, [2.5, 97.5])
    return rho, ci[0], ci[1], n, len(rhos), n_skip

robustness_rows = []

# Main spec
rho, lo, hi, n, nvalid, nskip = spearman_with_ci(q1_full["transport_share"].values, q1_full["pace_pp"].values)
robustness_rows.append({"spec": "Main: cars, EDGAR 2025, 2018->2024",
                        "n": n, "rho": rho, "ci_lo": lo, "ci_hi": hi,
                        "boot_valid": nvalid})

# Robustness A: EDGAR 2024
ts_2024 = edgar_transport_share(2024)
q1_2024 = q1_full.copy()
q1_2024["transport_share"] = ts_2024.reindex(q1_full.index)
rho, lo, hi, n, nvalid, nskip = spearman_with_ci(q1_2024["transport_share"].values, q1_2024["pace_pp"].values)
robustness_rows.append({"spec": "Robustness A: cars, EDGAR 2024, 2018->2024",
                        "n": n, "rho": rho, "ci_lo": lo, "ci_hi": hi,
                        "boot_valid": nvalid})

# Robustness B: 2 and 3 wheelers (2018 -> 2024)
ev_2w = ev[(ev["parameter"] == "EV sales share") & (ev["mode"] == "2 and 3 wheelers")
            & (ev["powertrain"] == "EV")].copy()
level_2w_18 = ev_2w[ev_2w["year"] == 2018].set_index("country")["value"]
level_2w_24 = ev_2w[ev_2w["year"] == 2024].set_index("country")["value"]
pace_2w = (level_2w_24 - level_2w_18).rename("pace_2w")
q1_2w = q1_full.join(pace_2w, how="inner")
rho, lo, hi, n, nvalid, nskip = spearman_with_ci(q1_2w["transport_share"].values, q1_2w["pace_2w"].values)
robustness_rows.append({"spec": "Robustness B: 2&3 wheelers, EDGAR 2025, 2018->2024",
                        "n": n, "rho": rho, "ci_lo": lo, "ci_hi": hi,
                        "boot_valid": nvalid})

# Robustness C: 2015 -> 2024 window (cars; Vietnam may be missing)
ev_cars_15 = ev[(ev["parameter"] == "EV sales share") & (ev["mode"] == "Cars")
                & (ev["powertrain"] == "EV") & (ev["year"] == 2015)]
if len(ev_cars_15) > 0:
    level_15 = ev_cars_15.set_index("country")["value"]
    level_24b = ev_sales[ev_sales["year"] == 2024].set_index("country")["value"]
    pace_15 = (level_24b - level_15).dropna().rename("pace_15")
    q1_15 = q1_full.join(pace_15, how="inner")
    rho, lo, hi, n, nvalid, nskip = spearman_with_ci(q1_15["transport_share"].values, q1_15["pace_15"].values)
    robustness_rows.append({"spec": "Robustness C: cars, EDGAR 2025, 2015->2024",
                            "n": n, "rho": rho, "ci_lo": lo, "ci_hi": hi,
                            "boot_valid": nvalid})
    print(f"Robustness C: {len(pace_15)} countries have 2015 cars data "
          f"(missing: {sorted(set(q1_full.index) - set(pace_15.index))})")
else:
    print("No 2015 cars data available; skipping 2015 window for cars.")

# Robustness D: 2015 -> 2024 window (2 and 3 wheelers)
ev_2w_15 = ev[(ev["parameter"] == "EV sales share") & (ev["mode"] == "2 and 3 wheelers")
              & (ev["powertrain"] == "EV") & (ev["year"] == 2015)]
if len(ev_2w_15) > 0:
    level_2w_15 = ev_2w_15.set_index("country")["value"]
    level_2w_24b = ev_2w[ev_2w["year"] == 2024].set_index("country")["value"]
    pace_2w_15 = (level_2w_24b - level_2w_15).dropna().rename("pace_2w_15")
    q1_2w_15 = q1_full.join(pace_2w_15, how="inner")
    rho, lo, hi, n, nvalid, nskip = spearman_with_ci(q1_2w_15["transport_share"].values, q1_2w_15["pace_2w_15"].values)
    robustness_rows.append({"spec": "Robustness D: 2&3 wheelers, EDGAR 2025, 2015->2024",
                            "n": n, "rho": rho, "ci_lo": lo, "ci_hi": hi,
                            "boot_valid": nvalid})
    print(f"Robustness D: {len(pace_2w_15)} countries have 2015 2&3-wheeler data "
          f"(missing: {sorted(set(q1_full.index) - set(pace_2w_15.index))})")

robust_df = pd.DataFrame(robustness_rows)
print("\nRobustness summary (Spearman rho, 95% bootstrap CI):")
display(robust_df.round(3))


Robustness C: 6 countries have 2015 cars data (missing: ['Vietnam'])
Robustness D: 6 countries have 2015 2&3-wheeler data (missing: ['Japan'])

Robustness summary (Spearman rho, 95% bootstrap CI):


,spec,n,rho,ci_lo,ci_hi,boot_valid
0,"Main: cars, EDGAR 2025, 2018->2024",7,-0.571,-1.0,0.373,10000
1,"Robustness A: cars, EDGAR 2024, 2018->2024",7,-0.571,-1.0,0.373,10000
2,"Robustness B: 2&3 wheelers, EDGAR 2025, 2018->...",7,-0.750,-1.0,0.059,10000
3,"Robustness C: cars, EDGAR 2025, 2015->2024",6,-0.371,-1.0,0.806,9999
4,"Robustness D: 2&3 wheelers, EDGAR 2025, 2015->...",6,-0.657,-1.0,0.500,9999


In [11]:
# --- Interpretation (explicit numeric answer) ---
rho_main = stats.spearmanr(q1_full["transport_share"], q1_full["pace_pp"])[0]
n_main = len(q1_full)
# recompute CI from stored? recompute quickly
rng = np.random.default_rng(SEED)
rhos_main = []
for _ in range(10000):
    idx = rng.integers(0, n_main, size=n_main)
    r, _ = stats.spearmanr(q1_full["transport_share"].values[idx],
                            q1_full["pace_pp"].values[idx])
    rhos_main.append(r)
ci_main = np.percentile(rhos_main, [2.5, 97.5])

print("=== Q1 — Explicit answer ===")
print(f"Spearman rho (transport share vs adoption pace, n={n_main}) = {rho_main:.4f}")
print(f"95% bootstrap CI = [{ci_main[0]:.4f}, {ci_main[1]:.4f}]")
print(f"Exact permutation p (two-sided, 5040 reps) = {p_exact:.4f}")
print(f"Pre-registered threshold |rho| >= {SPEARMAN_CRIT:.4f} (0.786): "
      f"{'MET' if abs(rho_main) >= SPEARMAN_CRIT else 'NOT MET'}")
print(f"Bootstrap CI excludes 0: {(ci_main[0] > 0) or (ci_main[1] < 0)}")
print()
print("Income control (OLS, log GDP 2018):")
print(f"  transport_share coef = {model_2018.params['transport_share']:.4f}, "
      f"95% CI = {model_2018.conf_int().loc['transport_share'].round(4).tolist()}, "
      f"p = {model_2018.pvalues['transport_share']:.4f}")
print()
print("Conclusion: with n=7, the rank association is "
      f"{'distinguishable from zero' if (abs(rho_main) >= SPEARMAN_CRIT and ((ci_main[0]>0) or (ci_main[1]<0))) else 'NOT distinguishable from zero'}.")
print("A null result here is NOT evidence of no relationship (SPECIFICATION S5).")
print("No causal language used (SPECIFICATION S7.5).")
print()
print("Key contrast: China (transport share ~8.3%, fastest adoption) vs "
      "New Zealand (transport share ~52.7%).")


=== Q1 — Explicit answer ===
Spearman rho (transport share vs adoption pace, n=7) = -0.5714
95% bootstrap CI = [-1.0000, 0.3725]
Exact permutation p (two-sided, 5040 reps) = 0.2000
Pre-registered threshold |rho| >= 0.7857 (0.786): NOT MET
Bootstrap CI excludes 0: False

Income control (OLS, log GDP 2018):
  transport_share coef = -0.4615, 95% CI = [-1.6224, 0.6994], p = 0.3317

Conclusion: with n=7, the rank association is NOT distinguishable from zero.
A null result here is NOT evidence of no relationship (SPECIFICATION S5).
No causal language used (SPECIFICATION S7.5).

Key contrast: China (transport share ~8.3%, fastest adoption) vs New Zealand (transport share ~52.7%).


## Q3 — Charging density and adoption (Dat)

### Preprocessing (charging density, 1-year lag)

### Analysis (PanelOLS, lead/lag check, wild cluster bootstrap)

### Results

### Interpretation

In [12]:
# Q3 code here

## Q3b — US tax-credit event study (future work)

Placeholder. Hoãn sang future work (ADR-006 D.7).

In [13]:
# Q3b placeholder

## Q4 — Vietnam two-wheeler transition (Khoi)

Design pinned by ADR-008. Four factors, each labelled **data** or **assumption**
as SPECIFICATION §3 Q4 requires:

| # | Factor | Value | Kind |
| --- | --- | --- | --- |
| 1 | Vietnam transport CO₂ | from EDGAR 2025 | **data** |
| 2 | Two-wheeler share of road transport | 50% / 65% / 80% | **assumption** — with road = 100% of EDGAR `Transport`, so this is a ceiling (D2) |
| 3 | Electrification of the two-wheeler fleet | `EV stock share`, 2&3 wheelers, 2024 | **data** |
| 4 | 1 − (EV intensity / ICE intensity) | grid 0.60 / 0.45 / 0.30 kgCO₂/kWh, ICE 2.5 L/100km, EV 35 Wh/km | **assumption** (D1) |

Decisions taken 2026-11-26: D1 grid-only headline with a joint envelope as robustness ·
D2 no road split, explicit ceiling · D3 2024 anchor plus a 100% counterfactual ·
D4 2&3 wheelers primary per SPECIFICATION §3, base-rate argument in the discussion.

In [ ]:
# --- Q4: Vietnam series (2 and 3 wheelers primary, cars for comparison) ---
VN = "Vietnam"
YEARS = list(range(2014, 2025))

def vn_series(param, mode):
    s = ev[(ev["country"] == VN) & (ev["parameter"] == param)
           & (ev["mode"] == mode)].set_index("year")["value"]
    return pd.Series([s.get(y, np.nan) for y in YEARS], index=YEARS, dtype=float)

bike_sales = vn_series("EV sales share", "2 and 3 wheelers")
car_sales = vn_series("EV sales share", "Cars")
bike_stock = vn_series("EV stock share", "2 and 3 wheelers")
car_stock = vn_series("EV stock share", "Cars")

vn_tab = pd.DataFrame({"bike_sales": bike_sales, "car_sales": car_sales,
                       "bike_stock": bike_stock, "car_stock": car_stock}).round(4)
print("Vietnam, % (NaN = year absent from the source, NOT interpolated):")
print(vn_tab.where(vn_tab.notna(), "—").to_string())

print("\nnon-null years:", {
    "bike_sales": int(bike_sales.notna().sum()),
    "car_sales": int(car_sales.notna().sum()),
    "bike_stock": int(bike_stock.notna().sum()),
    "car_stock": int(car_stock.notna().sum()),
})

car_missing = car_sales[car_sales.isna()].index.tolist()
print("cars sales share missing for:", car_missing,
      "(ADR-008 D.3 requires this gap be drawn, not interpolated)")

### The gap — and the crossover

The 2&3 wheeler series covers all 11 years. The `cars` series does **not**: it is
absent for 2014–2017 and again for 2019–2020. ADR-008 D.3 requires that gap to be
**drawn**, not interpolated, so the car line breaks rather than being joined.

Watch 2023–2024: it is where the two series cross.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

for ax, bike, car, label in [(axes[0], bike_sales, car_sales, "sales share"),
                             (axes[1], bike_stock, car_stock, "stock share")]:
    ax.plot(np.arange(len(YEARS)), bike.values, "o-", lw=2, ms=5,
            color="#1f77b4", label="2 and 3 wheelers")
    # cars: keep NaN so the line BREAKS instead of interpolating across 2019-2020
    ax.plot(np.arange(len(YEARS)), car.values, "s--", lw=2, ms=5,
            color="#ff7f0e", label="Cars")
    ax.set_xticks(np.arange(len(YEARS)))
    ax.set_xticklabels(YEARS, rotation=45)
    ax.set_xlabel("year")
    ax.set_ylabel(f"EV {label} (%)")
    ax.set_title(f"Vietnam — EV {label}")
    ax.grid(alpha=0.3)
    ax.legend()

# mark the crossover
axes[0].axvline(YEARS.index(2023), color="grey", ls=":", lw=1)
axes[0].annotate("cars overtake", xy=(YEARS.index(2023), car_sales.get(2023)),
                 xytext=(YEARS.index(2019) + 0.4, 12), fontsize=9,
                 arrowprops=dict(arrowstyle="->", color="grey"))

plt.tight_layout()
plt.show()

print("crossover years (cars above bikes):")
for metric, bike, car in [("sales share", bike_sales, car_sales),
                          ("stock share", bike_stock, car_stock)]:
    both = pd.DataFrame({"bike": bike, "car": car}).dropna()
    above = [int(y) for y, r in both.iterrows() if r["car"] > r["bike"]]
    print(f"  {metric}: {above if above else 'none'}")

### The band — ADR-008 D.4

```
CO₂ avoided = transport CO₂
            × two-wheeler share of road transport      (50/65/80, ceiling: road = 100%)
            × EV stock share of the two-wheeler fleet  (2024, data)
            × (1 − EV intensity / ICE intensity)       (grid 0.60/0.45/0.30)
```

Three rules from ADR-008 apply: report a **band**, never a single number; state each
factor as data or assumption; and do not present `transport CO₂ × share` as a result —
that is the industry size, not avoidable emissions.

In [ ]:
# --- factor 1: data ---
vn_em = emissions[emissions["country"] == VN]
TRANSPORT_MT = float(vn_em.loc[vn_em["sector"] == "Transport", "emissions_mt"].sum())
TOTAL_MT = float(vn_em["emissions_mt"].sum())
print(f"transport CO2 = {TRANSPORT_MT:.3f} Mt | national total = {TOTAL_MT:.3f} Mt "
      f"| transport share = {TRANSPORT_MT/TOTAL_MT*100:.2f}%")

# --- factor 3: data ---
STOCK_SHARE = float(bike_stock.loc[2024]) / 100.0
print(f"EV stock share, 2 and 3 wheelers, 2024 = {STOCK_SHARE*100:.2f}%")

# --- factor 4: assumption ---
PETROL_KG_PER_L = 2.31          # kg CO2 per litre of petrol
ICE_L_PER_100KM = 2.5           # assumption
EV_KWH_PER_KM = 0.035           # assumption
GRID = {"high-carbon": 0.60, "central": 0.45, "low-carbon": 0.30}   # kg CO2/kWh

def intensity_factor(grid_kg_per_kwh, ice_l=ICE_L_PER_100KM, ev_kwh=EV_KWH_PER_KM):
    ice_g_per_km = ice_l / 100.0 * PETROL_KG_PER_L * 1000.0
    ev_g_per_km = ev_kwh * grid_kg_per_kwh * 1000.0
    return 1.0 - ev_g_per_km / ice_g_per_km

print("\nintensity factor (1 - EV/ICE):")
for name, g in GRID.items():
    print(f"  {name:12s} grid={g:.2f} kgCO2/kWh -> factor {intensity_factor(g):.4f}")

# --- factor 2: assumption (ceiling: road = 100% of EDGAR Transport) ---
SHARES = {"low": 0.50, "central": 0.65, "high": 0.80}

rows = []
for s_name, s in SHARES.items():
    for g_name, g in GRID.items():
        mt = TRANSPORT_MT * s * STOCK_SHARE * intensity_factor(g)
        rows.append({"share": s_name, "share_pct": s * 100,
                     "grid": g_name, "factor": round(intensity_factor(g), 4),
                     "mt": mt, "pct_national": mt / TOTAL_MT * 100})

band = pd.DataFrame(rows).sort_values("mt")
print("\n=== 2024 band: 3 shares x 3 grid points (road = 100%, ceiling) ===")
print(band.round(4).to_string(index=False))

lo, md_, hi = band["mt"].min(), band["mt"].median(), band["mt"].max()
print(f"\nBAND  min {lo:.4f} Mt | median {md_:.4f} Mt | max {hi:.4f} Mt")
print(f"      {lo/TOTAL_MT*100:.3f}% | {md_/TOTAL_MT*100:.3f}% | {hi/TOTAL_MT*100:.3f}% of national CO2")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
colors = {"low-carbon": "#2ca02c", "central": "#7f7f7f", "high-carbon": "#d62728"}
for g_name in GRID:
    sub = band[band["grid"] == g_name].sort_values("share_pct")
    ax.plot(sub["share_pct"], sub["mt"], "o-", color=colors[g_name], lw=2, label=f"grid {g_name}")
ax.axhline(md_, color="k", ls=":", lw=1, label=f"median {md_:.2f} Mt")
ax.set_xlabel("assumed two-wheeler share of road transport (%)")
ax.set_ylabel("CO₂ avoided (Mt)")
ax.set_title(f"Vietnam 2024 — avoided CO₂ band\n(EV stock share {STOCK_SHARE*100:.1f}%, ceiling: road = 100% of Transport)")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### Robustness — joint envelope over grid **and** efficiency

D1 keeps the headline grid-only (one axis, as SPECIFICATION §5 says). This widens
the assumption space to ICE consumption and EV efficiency as well, to show how much
of the band's width comes from the fourth factor rather than from the 50/65/80 share.

In [ ]:
ICE_OPTS = [2.0, 2.5, 3.0]      # L/100km
EV_OPTS = [0.040, 0.035, 0.030]  # kWh/km

rows = []
for s_name, s in SHARES.items():
    for ice in ICE_OPTS:
        for evk in EV_OPTS:
            for g_name, g in GRID.items():
                f = intensity_factor(g, ice, evk)
                mt = TRANSPORT_MT * s * STOCK_SHARE * f
                rows.append({"share": s_name, "ice": ice, "ev_kwh": evk,
                             "grid": g_name, "factor": round(f, 4), "mt": mt})
env = pd.DataFrame(rows)
print(f"combinations: {len(env)}")
print(f"factor range: {env.factor.min():.4f} - {env.factor.max():.4f}")
print(f"BAND  min {env.mt.min():.4f} | median {env.mt.median():.4f} | max {env.mt.max():.4f} Mt")
print(f"      {env.mt.min()/TOTAL_MT*100:.3f}% | {env.mt.median()/TOTAL_MT*100:.3f}% | "
      f"{env.mt.max()/TOTAL_MT*100:.3f}% of national CO2")

print("\nwhich assumption drives the width?")
print(f"  intensity factor spans {env.factor.max()/env.factor.min():.2f}x")
print(f"  two-wheeler share spans {0.80/0.50:.2f}x")
print(f"  -> combined envelope is roughly {(env.mt.max()/env.mt.min()):.2f}x the smallest value")

### The counterfactual ceiling (D3)

"How much does it avoid now" and "how much could it ever avoid" are different
questions, and only the pair makes the 2024 number interpretable.

Second block: raise the two-wheeler fleet's EV stock share to **100%**, leaving
everything else alone. That is the maximum this segment could ever deliver, so it
bounds every future claim.

In [ ]:
rows = []
for s_name, s in SHARES.items():
    for g_name, g in GRID.items():
        f = intensity_factor(g)
        mt = TRANSPORT_MT * s * 1.0 * f          # 100% electrified two-wheeler fleet
        rows.append({"share": s_name, "grid": g_name, "factor": round(f, 4),
                     "mt": mt, "pct_national": mt / TOTAL_MT * 100})
cf = pd.DataFrame(rows).sort_values("mt")
print("=== counterfactual: EV stock share = 100% (not a forecast, a bound) ===")
print(cf.round(3).to_string(index=False))
print(f"\nCEILING  min {cf.mt.min():.2f} Mt ({cf.pct_national.min():.2f}% of national CO2) "
      f"| max {cf.mt.max():.2f} Mt ({cf.pct_national.max():.2f}%)")
print(f"\nfor scale: today's 2024 band is {md_:.3f} Mt, "
      f"i.e. {md_/cf.mt.median()*100:.1f}% of the way to that ceiling")

### Why the share metric misleads (D4) — for the discussion

The crossover in 2023–24 says **cars electrified faster from a smaller base**. It does
not say cars matter more. Motorbikes are a per-person purchase; cars are a per-family
purchase behind heavy taxes:

- **45 million registered motorbikes** against a 92 million population; **87% of
  households** own one — among the highest ownership rates worldwide.
- **2 million registered passenger cars** (2015), with import and sales taxes able to
  make up 2–3× the final price; sales growing double-digit annually.

So the fleet ratio was roughly **22:1**. Applying the 2024 shares to that ratio:

```
bikes: 2.6% of 22 units  ≈ 0.57 units of EV stock
cars : 4.6% of  1 unit   ≈ 0.05 units of EV stock     ->  ~12x more electric bikes
```

**Share is base-rate blind.** Cars lead on share while bikes still carry roughly an
order of magnitude more electric vehicles in absolute terms.

⚠️ Two honest limits on that number: the 22:1 is the **2015** fleet (car sales have
grown since, so the true ratio is lower), and GEVO has **no absolute EV counts for
Vietnam**, so this is an illustration of the mechanism, not a measurement from our data.

*This paragraph is a draft of the argument, not the final prose — the author writes that.*

### Limitations

- **EDGAR cannot isolate two-wheelers.** Its `Transport` sector contains road,
  domestic aviation, shipping and rail. With road = 100% (D2), factor 2 is a **ceiling**,
  not an estimate.
- **No absolute EV counts for Vietnam in GEVO** — shares only. Fleet dominance can be
  illustrated from external figures but cannot be settled from this dataset.
- **The 2&3 wheeler series is aggregate**, covering both two- and three-wheelers, and
  the `cars` series has genuine gaps (2014–17, 2019–20) that are drawn, not filled.
- **The intensity factor depends on grid intensity**, which is a scenario input, not a
  measurement of Vietnam's grid in 2024.
- The band is small under every assumption in the envelope; SPECIFICATION §5 says to
  state that plainly rather than search for a metric that makes it larger.

### Interpretation

*Your words here.* SPECIFICATION §5 Q4 requires: always report a band; if the band is
too wide to conclude anything, say so plainly; never use "impact", only "falls in the
range". Report the 2024 band against the 100% counterfactual so the small number is
interpretable rather than merely small.